In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import os

# ====================== Định nghĩa đường dẫn ======================
BASE_DIR = os.path.dirname(os.getcwd())   # Trỏ về thư mục gốc project
DATA_PATH = os.path.join(BASE_DIR, 'data/')

print("Đang làm việc tại:", BASE_DIR)

# ====================== Load dữ liệu ======================
df = pd.read_csv(DATA_PATH + 'WA_Fn-UseC_-Telco-Customer-Churn.csv')

# ====================== Preprocessing ======================
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna().reset_index(drop=True)

# Drop cột không cần
df = df.drop(['customerID'], axis=1)

# Encode categorical columns
cat_cols = df.select_dtypes(include=['object']).columns.drop('Churn')
le_dict = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le

# Target variable
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print("Shape sau preprocessing:", df.shape)

# ====================== Feature Engineering ======================
df['AvgMonthlyCharge'] = df['TotalCharges'] / (df['tenure'] + 1)
df['HighCharge'] = (df['MonthlyCharges'] > df['MonthlyCharges'].quantile(0.75)).astype(int)
df['LowTenureHighCharge'] = ((df['tenure'] < 12) & (df['HighCharge'] == 1)).astype(int)

# One-hot encoding cho TenureGroup
df = pd.get_dummies(df, columns=['TenureGroup'], drop_first=True) if 'TenureGroup' in df.columns else df

print("\nFeatures sau engineering:", df.columns.tolist())

# ====================== Train-Test Split ======================
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numerical features
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlyCharge']
scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

# ====================== Lưu dữ liệu ======================
X_train.to_csv(DATA_PATH + 'X_train.csv', index=False)
X_test.to_csv(DATA_PATH + 'X_test.csv', index=False)
y_train.to_csv(DATA_PATH + 'y_train.csv', index=False)
y_test.to_csv(DATA_PATH + 'y_test.csv', index=False)

print("\n✅ Preprocessing & Feature Engineering HOÀN TẤT!")
print(f"Đã lưu file vào: {DATA_PATH}")
print("Train shape:", X_train.shape)

Đang làm việc tại: d:\UIT_SDH\Ky_3\thay thuan\telco-customer-churn-prediction
Shape sau preprocessing: (7032, 20)

Features sau engineering: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'AvgMonthlyCharge', 'HighCharge', 'LowTenureHighCharge']

✅ Preprocessing & Feature Engineering HOÀN TẤT!
Đã lưu file vào: d:\UIT_SDH\Ky_3\thay thuan\telco-customer-churn-prediction\data/
Train shape: (5625, 22)
